In [148]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import root_mean_squared_error, r2_score
import joblib

In [149]:
df = pd.read_csv('../data/engineered_training_data.csv', parse_dates=['Time'])
df = df.set_index('Time')
df.columns

Index(['GHI(W/m2)', 'Windspeed(m/s)', 'Solar Energy(MWh)', 'Wind Energy(MWh)',
       'Total Energy(MWh)', 'Battery Charge(MWh)', 'Battery Discharge(MWh)',
       'Stored Energy(MWh)', 'Energy To Grid(MWh)', 'Hydrogen_yield(kg)', 'Hr',
       'Mon', 'Day', 'spline_hr_1', 'spline_hr_2', 'spline_hr_3',
       'spline_hr_4', 'Windspeed_mean_3h', 'Windspeed_std_3h', 'GHI_mean_3h',
       'Windspeed_lag_1hr', 'GHI_lag_1hr'],
      dtype='str')

In [150]:
y_train = df['Hydrogen_yield(kg)']

# Using only the core weather features adding time and battery-relaated features will cause
# overfitting due to polynomial expansion which results in noise increasing the adjusted R2 score
# These 2 features give one of the highest Adjusted R2 scores further proving that keeping
# a baseline of these core features is the correct move
poly_features = ['Windspeed(m/s)' , 'GHI(W/m2)']
X_train_poly = df[poly_features]

print(f'No. of rows and columns in x: {X_train_poly.shape}')
print(f'No. of rows in y: {y_train.shape[0]}')

No. of rows and columns in x: (96432, 2)
No. of rows in y: 96432


In [151]:
# Wind Energy follows a cubic rule (windspeed ^ 3) which is why degree 3 works here the best
# Though a smooth polynomial curve does struggle when a cap of 9.5MW is kept prompting to try different models
poly = PolynomialFeatures(degree=3, include_bias=False)
X_train_poly_trans = poly.fit_transform(X_train_poly)

poly_model = LinearRegression()
poly_model.fit(X_train_poly_trans, y_train)

y_train_pred = poly_model.predict(X_train_poly_trans)
rmse = root_mean_squared_error(y_train, y_train_pred)
r2 = r2_score(y_train, y_train_pred)

# Adjusted R2 score to observe the features which are noise
n = X_train_poly_trans.shape[0]
p = X_train_poly_trans.shape[1]
adjusted_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))

print(f'Transformed Feature Count: {X_train_poly_trans.shape[1]}')
print(f'RMSE: {rmse:.3f} kg')
print(f'R2 Score: {r2:.4f}')
print(f'Adjusted R2 Score: {adjusted_r2:.4f}')

Transformed Feature Count: 9
RMSE: 18.524 kg
R2 Score: 0.8926
Adjusted R2 Score: 0.8926


In [152]:
# Exported both transformer and model to use in testing and deployment phase
joblib.dump(poly, '../models/poly_trans.joblib')
joblib.dump(poly_model, '../models/poly_reg_model.joblib')

['../models/poly_reg_model.joblib']

In [154]:
# Training Results of Random Forest Regressor
# Pulling the model from the script we wrote in train.py
rf_model = joblib.load('../models/random_forest_regressor.joblib')
# Using these exact set of features since using Solar, wind and total energy 
# will make the model overfitted and memorize rather than learning patterns
features = ['GHI(W/m2)', 'Windspeed(m/s)', 'Stored Energy(MWh)', 'Mon',
            'Day', 'spline_hr_1','spline_hr_2', 'spline_hr_3',
            'spline_hr_4', 'Windspeed_mean_3h','Windspeed_std_3h',
            'GHI_mean_3h', 'Windspeed_lag_1hr','GHI_lag_1hr'
            ]
X_train_rf = df[features]
y_train_rf = rf_model.predict(X_train_rf)

rf_rmse = root_mean_squared_error(y_train, y_train_rf)
rf_r2 = r2_score(y_train, y_train_rf)
n_rf = X_train_rf.shape[0]
p_rf = X_train_rf.shape[1]
adjusted_rf_r2 = 1 - ((1 - rf_r2) * (n_rf - 1) / (n_rf - p_rf - 1))

print('Random Forest:')
print(f'RMSE: {rf_rmse:.4f}kg')
print(f'R2 Score: {rf_r2:.4f}')
print(f'Adjusted R2: {adjusted_rf_r2:.4f}')

Random Forest:
RMSE: 3.4184kg
R2 Score: 0.9963
Adjusted R2: 0.9963
